# Gerar Legenda MULTICOR — 5 Idiomas (classificação gramatical por palavra)

> 🖥️ **CPU basta** — este notebook não usa GPU. Deixe o acelerador em *Nenhum*: não muda nada aqui e poupa sua cota de GPU, que é limitada.

Gera o arquivo `.ass` com a legenda colorida — não queima em nenhum vídeo
ainda (isso é o notebook `caption-multicolor-burn.ipynb`, separado de
propósito, pra dar espaço pra correção manual no meio do caminho).

No final, duas ações **separadas**: baixar o `.ass` pra revisar/corrigir,
e (só depois de confirmar que ficou bom) salvar no Drive.

## O Stanza e o Kiwi erram, e o erro deles é mudo

Sai uma cor plausível e ninguém percebe até assistir o vídeo. São três
camadas contra isso, em ordem de quanto poupam de trabalho seu:

**Célula 4 — correção automática.** O que já se sabe que erra, conserta
sozinho: a invariante da pontuação e o léxico versionado
(`dados_lexico/classes-excecoes.json`). Como é versionado, correção feita uma
vez vale pra todo capítulo seguinte.

**Célula 5 — revisar.** Aponta ONDE olhar, em vez de pedir que você leia
3.300 peças. No Mateus 2 são 39 apontamentos (1,2%). Saem dois arquivos:

| arquivo | pra quê |
|---|---|
| `<nome>_classes_revisar.html` | **achar** o erro — a legenda pintada com as cores de verdade, os 5 idiomas empilhados como no vídeo |
| `<nome>_classes_revisar.csv` | **corrigir** — abre no Sheets, você muda a coluna `classe` |

**Célula 6 — aplicar.** Lê o CSV de volta, recusa classe que não existe, e
sugere quais das suas correções merecem virar regra automática (só as que
você repetiu — correção de uma vez só costuma ser contexto).

Se a revisão não achou nada, pule a 6 e vá direto pra 7.


In [5]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  1. SETUP                                                        ║
# ╚══════════════════════════════════════════════════════════════════╝
!pip install -q stanza kiwipiepy

import shutil, sys
from pathlib import Path

import stanza
from kiwipiepy import Kiwi
from google.colab import drive

try:
    drive.flush_and_unmount()
except Exception:
    pass
drive.mount('/content/drive', force_remount=True)

PASTA_DRIVE_RAIZ_MODULOS = "narrated_video"
PASTA_MODULOS = Path(f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ_MODULOS}/pipeline/modulos")
DESTINO = Path("/content/pipeline")
if PASTA_MODULOS.exists():
    if DESTINO.exists():
        shutil.rmtree(DESTINO)
    shutil.copytree(PASTA_MODULOS, DESTINO)
    print(f"✅ {len(list(DESTINO.glob('*.py')))} módulos copiados")

    # ── A cópia trouxe TODOS os módulos? ───────────────────────────────────────
    # "N módulos copiados" sozinho não quer dizer nada. E o modo de falhar aqui é
    # traiçoeiro: o Drive montado do Colab popula a listagem da pasta com atraso,
    # então um copytree logo depois do mount às vezes enxerga só parte dos
    # arquivos. Já aconteceu de copiar 13 de 31 -- com visto verde -- e o notebook
    # quebrar muito depois, num import, longe da causa.
    #
    # A conferência é de três pontas, porque a causa muda o conserto:
    #   manifesto  o que o repositório tem  (versionado; chega pela cópia)
    #   Drive      o que chegou lá
    #   VM         o que a cópia desta célula trouxe
    # A conferência tem duas perguntas, e SÓ UMA delas precisa do manifesto:
    #
    #   Drive → VM   a cópia acima trouxe tudo?      dá pra ver aqui mesmo
    #   repo → Drive o Drive está em dia?            só o manifesto sabe
    #
    # A versão anterior amarrava as duas ao manifesto: sem ele, imprimia um
    # aviso e seguia SEM CONFERIR NADA. Foi assim que "✅ 13 modules copied"
    # passou com visto verde num Drive que tinha 31 -- justamente no dia em
    # que o manifesto ainda não existia. Comparar 13 com 31 nunca dependeu de
    # manifesto nenhum.
    _no_drive = {f.name for f in PASTA_MODULOS.glob("*.py")}
    _na_vm    = {f.name for f in DESTINO.glob("*.py")}

    # ── Drive → VM ────────────────────────────────────────────────────────
    # O Drive montado do Colab popula a listagem da pasta com atraso, então um
    # copytree logo depois do mount às vezes enxerga só parte dos arquivos.
    # Uma segunda passada, com o mount já quente, costuma resolver.
    _nao_copiados = sorted(_no_drive - _na_vm)
    if _nao_copiados:
        print(f"   ⏳ {len(_nao_copiados)} módulo(s) não vieram na 1ª passada — copiando de novo")
        for _n in _nao_copiados:
            shutil.copyfile(PASTA_MODULOS / _n, DESTINO / _n)
        _na_vm = {f.name for f in DESTINO.glob("*.py")}
        _nao_copiados = sorted(_no_drive - _na_vm)
    if _nao_copiados:
        print(f"\n🚨 {len(_nao_copiados)} módulo(s) estão no Drive mas não copiaram:")
        for _n in _nao_copiados:
            print(f"     {_n}")
        raise SystemExit("Rode ESTA célula de novo — o Drive montado ainda estava acordando.")
    print(f"   ✅ os {len(_no_drive)} módulos do Drive chegaram na VM")

    # ── repositório → Drive ───────────────────────────────────────────────
    _manifesto = PASTA_MODULOS / "_manifesto.txt"
    if not _manifesto.exists():
        print("   ⚠️  sem _manifesto.txt: não dá pra saber se o DRIVE está atrás")
        print("      do repositório. Ele é versionado — rode o repositorio-sincronizar.")
    else:
        _esperados = {l.strip() for l in _manifesto.read_text().splitlines()
                      if l.strip() and not l.startswith("#")}
        _fora_do_drive = sorted(_esperados - _no_drive)
        if _fora_do_drive:
            print(f"\n🚨 {len(_fora_do_drive)} módulo(s) não estão no DRIVE:")
            for _n in _fora_do_drive:
                print(f"     {_n}")
            raise SystemExit("Rode o repositorio-sincronizar.ipynb — o Drive está atrás do repositório.")
        print(f"   ✅ e batem com os {len(_esperados)} do manifesto")

    # ── O Python está segurando a versão anterior? ────────────────────────
    # Copiar arquivo novo por cima não desfaz um import já feito: o Python
    # guarda o módulo em sys.modules e reaproveita. Numa sessão longa, isso
    # faz o notebook rodar com o config.py de ontem mesmo depois de um sync
    # perfeito -- e o sintoma aparece longe da causa (nome de arquivo que
    # mudou, padrão que era pra ter mudado e não mudou). Descarregar aqui
    # equivale a reiniciar o runtime, sem perder o resto da sessão.
    _recarregar = [_n for _n, _m in list(sys.modules.items())
                   if getattr(_m, "__file__", None) and str(DESTINO) in str(_m.__file__)]
    for _n in _recarregar:
        del sys.modules[_n]
    if _recarregar:
        print(f"   ♻️  {len(_recarregar)} módulo(s) já importados foram descarregados —")
        print(f"      o import vai reler a cópia nova (rode as células seguintes de novo)")
else:
    print(f"❌ Pasta de módulos não encontrada: {PASTA_MODULOS}")
if str(DESTINO) not in sys.path:
    sys.path.insert(0, str(DESTINO))

# ── O ambiente combina com o que este notebook faz? ────────────────────────
# Cota de GPU do Colab é limitada e some sem aviso -- e parte da nossa foi
# gasta em notebook que não usa GPU pra nada, rodando com GPU só porque a
# seleção ficou de antes. Silencioso quando combina.
try:
    from ambiente import avisar_gpu
    avisar_gpu(precisa=False)
except Exception:
    pass

# ⚠️ Os módulos abaixo (classificacao.py, classificacao_ko.py, cores.py,
# renderizacao.py) precisam estar na MESMA pasta
# narrated_video/pipeline/modulos/ do Drive, junto com os antigos, pra esse
# copytree acima já trazer eles também. Se der erro de import na célula 4,
# é sinal de que faltou subir algum desses 4 arquivos pro Drive.

kiwi = Kiwi()
print("✅ Stanza e Kiwi prontos")


Mounted at /content/drive
✅ 18 módulos copiados
✅ Stanza e Kiwi prontos


In [6]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  2. CONFIGURAÇÃO                                                 ║
# ╚══════════════════════════════════════════════════════════════════╝
NOME_ORACAO = "40_Matt_02"
PASTA_DRIVE_RAIZ = "narrated_video"

IDIOMA_MESTRE = "en"  # idioma de referência — usa o SRT "whisper" (bruto, não
                      # sincronizado); os outros idiomas usam o conjunto já
                      # sincronizado pelo tempo do mestre (sem sufixo)

IDIOMAS_STANZA = {"pt": "pt", "en": "en", "es": "es", "fr": "fr"}  # idiomas que usam Stanza
IDIOMA_KIWI = "ko"  # idioma que usa Kiwi (só coreano, por enquanto)

BOX_BORDER = 6  # espessura da caixa colorida, em px

# Separador entre peças da MESMA palavra escrita — só o coreano tem isso, onde
# um morfema não pode ser separado do seguinte por um espaço normal sem quebrar
# a palavra. Com nada entre eles, as bordas de 6px encostam e as sílabas saem
# espremidas. "\u2009" é o espaço fino; "\u200a" é ainda mais estreito, e ""
# volta ao comportamento antigo (colado).
ESPACO_ENTRE_PECAS_COLADAS = "\u2009"

print(f"Vídeo: {NOME_ORACAO}")


Vídeo: 40_Matt_02


In [7]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  3. BAIXAR OS SRTs — mestre usa "whisper", os outros usam o          ║
# ║  conjunto JÁ SINCRONIZADO (sem sufixo, gerado pelo                ║
# ║  caption-multilang-generate.ipynb) — não usa mais o "whisper" bruto ║
# ║  dos outros idiomas, que tem timing próprio de cada dublagem e    ║
# ║  fica fora de sincronia por conteúdo.                             ║
# ╚══════════════════════════════════════════════════════════════════╗
from config import PipelineConfig
from drive_utils import DriveClient
from srt_utils import ler_srt

drive_client = DriveClient.get()
legendas_por_idioma_raw = {}

todos_idiomas = list(IDIOMAS_STANZA.keys()) + [IDIOMA_KIWI]
for idioma in todos_idiomas:
    config = PipelineConfig(NOME_ORACAO=NOME_ORACAO, PASTA_DRIVE_RAIZ=PASTA_DRIVE_RAIZ, IDIOMA_MESTRE=IDIOMA_MESTRE)

    # mestre = "whisper" (é a própria referência, não passa por sincronização);
    # os outros = sem sufixo (conjunto já sincronizado pelo tempo do mestre)
    if idioma == IDIOMA_MESTRE:
        nome_arquivo = config.nome_srt_whisper(idioma)
    else:
        nome_arquivo = config.nome_srt(idioma)

    destino_local = Path(nome_arquivo)
    ok = drive_client.download(config.pasta_oracao, nome_arquivo, destino_local)
    if not ok:
        print(f"  ⚠️  {idioma.upper()}: não achei '{nome_arquivo}' — pulando")
        continue
    legendas_por_idioma_raw[idioma] = ler_srt(destino_local)
    print(f"  ✅ {idioma.upper()} ({nome_arquivo}): {len(legendas_por_idioma_raw[idioma])} bloco(s)")

if not legendas_por_idioma_raw:
    raise FileNotFoundError("Nenhum SRT encontrado pra nenhum idioma")

if IDIOMA_MESTRE not in legendas_por_idioma_raw:
    print(f"⚠️  O idioma mestre ('{IDIOMA_MESTRE}') não foi encontrado — as legendas dos outros "
          f"idiomas estão sincronizadas em relação a ele, então isso é só um aviso informativo, "
          f"não impede o resto de rodar.")


  ✅ PT: 67 bloco(s)
  ✅ EN: 43 bloco(s)
  ✅ ES: 37 bloco(s)
  ✅ FR: 41 bloco(s)
  ✅ KO: 49 bloco(s)


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  4. CLASSIFICAR CADA BLOCO (Stanza pra PT/EN/ES/FR, Kiwi pro KO) ║
# ║  Se já existir uma classificação salva/corrigida no Drive        ║
# ║  (config.nome_classificacao_multicolor), usa ela em vez de rodar ║
# ║  o Stanza/Kiwi de novo pra esse idioma.                          ║
# ╚══════════════════════════════════════════════════════════════════╝
from classificacao import classificar_palavra_stanza
from classificacao_ko import classificar_pecas_palavra_ko
from renderizacao import (PecaColorida, salvar_classificacao_multicolor,
                          carregar_classificacao_multicolor, classificacao_confere)
from revisao_classes import aplicar_excecoes, carregar_lexico, corrigir_pontuacao

def _tentar_carregar_classificacao_salva(idioma):
    """Reaproveita a classificação salva no Drive -- mas só se ela for DESTA
    legenda. O reaproveitamento existe pra preservar correção manual; o que
    ele não pode fazer é ressuscitar o texto de uma versão anterior do SRT.
    As peças carregam o texto, então uma classificação velha faz o vídeo
    exibir as palavras antigas com o SRT novo parado ao lado."""
    nome_arquivo = config.nome_classificacao_multicolor(idioma)
    destino_local = Path(nome_arquivo)
    if not drive_client.download(config.pasta_oracao, nome_arquivo, destino_local):
        return None
    blocos = carregar_classificacao_multicolor(destino_local)
    divergencia = classificacao_confere(blocos, legendas_por_idioma_raw[idioma])
    if divergencia:
        print(f"   ⚠️  {idioma.upper()}: descartando '{nome_arquivo}' — {divergencia}.")
        print(f"      (é de uma versão anterior da legenda; vou classificar de novo)")
        return None
    return blocos

blocos_por_idioma = {}
idiomas_reaproveitados = []
idiomas_a_classificar = []

for idioma in list(IDIOMAS_STANZA.keys()) + [IDIOMA_KIWI]:
    if idioma not in legendas_por_idioma_raw:
        continue
    blocos_salvos = _tentar_carregar_classificacao_salva(idioma)
    if blocos_salvos is not None:
        blocos_por_idioma[idioma] = blocos_salvos
        idiomas_reaproveitados.append(idioma)
        print(f"✅ {idioma.upper()}: classificação já salva reaproveitada "
              f"({config.nome_classificacao_multicolor(idioma)}, {len(blocos_salvos)} bloco(s))")
    else:
        idiomas_a_classificar.append(idioma)

# ── Pipelines Stanza — só pros idiomas que ainda precisam ser classificados ──
pipelines_stanza = {}
for idioma, codigo in IDIOMAS_STANZA.items():
    if idioma not in idiomas_a_classificar:
        continue
    stanza.download(codigo, verbose=False)
    pipelines_stanza[idioma] = stanza.Pipeline(codigo, processors="tokenize,pos,lemma", verbose=False)
    print(f"✅ Pipeline Stanza pronto: {idioma}")

# ── PT / EN / ES / FR — todos iguais, palavra por palavra (a classificação
# ── simplificada não precisa mais de tratamento especial pro francês,
# ── já que substantivo não distingue mais gênero) ──────────────────────
for idioma in ("pt", "en", "es", "fr"):
    if idioma not in idiomas_a_classificar:
        continue
    nlp = pipelines_stanza[idioma]
    blocos = []
    for leg in legendas_por_idioma_raw[idioma]:
        doc = nlp(leg.texto)
        pecas = []
        for sentenca in doc.sentences:
            # TOKEN, não WORD. O Stanza segue a convenção do Universal
            # Dependencies e expande contração em palavras sintáticas: "da"
            # vira "de"+"a", "nos" vira "em"+"os", "del" vira "de"+"el", "du"
            # vira "de"+"le". Colorir por palavra faz a legenda EXIBIR essa
            # expansão -- "em os dias de o rei Herodes" no lugar de "nos dias
            # do rei Herodes". O texto que aparece na tela tem que ser o que
            # está escrito no SRT; a análise é só pra escolher a cor.
            tokens_da_sentenca = list(sentenca.tokens)
            for i, token in enumerate(tokens_da_sentenca):
                cabeca = token.words[0]   # na contração, a preposição
                # Duas regras precisam saber o que vem DEPOIS: "passou a
                # viver" tem preposição, "passou a fome" não. Sem esse
                # olhar adiante o Stanza devolve conjunção nos dois casos.
                proximo = (tokens_da_sentenca[i + 1].words[0]
                           if i + 1 < len(tokens_da_sentenca) else None)
                classe = classificar_palavra_stanza(
                    token.text, cabeca.lemma, cabeca.upos, cabeca.xpos,
                    cabeca.feats or "", idioma,
                    upos_seguinte=(proximo.upos if proximo else None),
                    feats_seguinte=((proximo.feats or "") if proximo else ""),
                )
                # O upos vai junto e NÃO entra na cor: é o que permite a
                # revisão distinguir "o Stanza errou" de "a regra mapeou
                # errado" -- que se consertam em lugares diferentes. Também
                # é como a revisão descobre a peça que caiu no fallback
                # ("adverbio" por falta de regra), hoje indistinguível de um
                # advérbio de verdade.
                pecas.append(PecaColorida(token.text, classe, upos=cabeca.upos))
        blocos.append({"inicio_ms": leg.inicio_ms, "fim_ms": leg.fim_ms, "pecas": pecas})
    blocos_por_idioma[idioma] = blocos

    # O que vai pra tela é o texto das PEÇAS, não o do SRT. Se a análise
    # devolver forma subjacente em vez do que está escrito, o vídeo sai com
    # texto errado e nada falha. Mesma conferência que guarda o cache.
    _divergencia = classificacao_confere(blocos, legendas_por_idioma_raw[idioma])
    if _divergencia:
        print(f"   🚩 {idioma.upper()}: a classificação NÃO reproduz o SRT — {_divergencia}.")
        print(f"      Não use este resultado sem olhar; o texto na tela sai diferente da legenda.")

    nome_arquivo = config.nome_classificacao_multicolor(idioma)
    salvar_classificacao_multicolor(blocos, Path(nome_arquivo))
    drive_client.upload(Path(nome_arquivo), config.pasta_oracao, "application/json")
    print(f"✅ {idioma.upper()} classificado: {len(blocos)} bloco(s) (salvo em {nome_arquivo})")

# ── COREANO — peça por peça, com colado_anterior pra não ter espaço dentro
# ── da mesma palavra original ────────────────────────────────────────────
if IDIOMA_KIWI in idiomas_a_classificar:
    blocos = []
    for leg in legendas_por_idioma_raw[IDIOMA_KIWI]:
        original = leg.texto
        tokens = kiwi.analyze(original)[0][0]
        grupos: dict[tuple, list] = {}
        ordem_grupos = []
        for t in tokens:
            chave = (t.sent_position, t.word_position)
            if chave not in grupos:
                grupos[chave] = []
                ordem_grupos.append(chave)
            grupos[chave].append(t)

        pecas = []
        for chave in ordem_grupos:
            toks = sorted(grupos[chave], key=lambda t: (t.start, -t.len))
            classes = classificar_pecas_palavra_ko(
                [{"peca": t.form, "classe_kiwi": t.tag} for t in toks])

            # O texto exibido vem SEMPRE de uma fatia do original, nunca de
            # t.form. Uma sílaba coreana pode conter dois morfemas -- "셨" é
            # 시+었 -- e o Kiwi devolve os dois separados: exibir a forma
            # escreveria "태어나시었을" onde está escrito "태어나셨을". Morfemas
            # que dividem a mesma fatia viram uma peça só, com a classe do
            # primeiro; onde não há contração, cada morfema continua com a
            # sua cor, que é o motivo de o coreano usar o Kiwi.
            fatias: list[list] = []
            for t, classe in zip(toks, classes):
                ini, fim = t.start, t.start + t.len
                if fatias and ini < fatias[-1][1]:
                    fatias[-1][1] = max(fatias[-1][1], fim)
                else:
                    fatias.append([ini, fim, classe, t.tag])
            for i, (ini, fim, classe, tag) in enumerate(fatias):
                texto_peca = original[ini:fim]
                if not texto_peca.strip():
                    continue
                pecas.append(PecaColorida(texto_peca, classe,
                                          colado_anterior=(i > 0), upos=tag))
        blocos.append({"inicio_ms": leg.inicio_ms, "fim_ms": leg.fim_ms, "pecas": pecas})
    blocos_por_idioma[IDIOMA_KIWI] = blocos

    _divergencia = classificacao_confere(blocos, legendas_por_idioma_raw[IDIOMA_KIWI])
    if _divergencia:
        print(f"   🚩 KO: a classificação NÃO reproduz o SRT — {_divergencia}.")
        print(f"      Não use este resultado sem olhar; o texto na tela sai diferente da legenda.")

    nome_arquivo = config.nome_classificacao_multicolor(IDIOMA_KIWI)
    salvar_classificacao_multicolor(blocos, Path(nome_arquivo))
    drive_client.upload(Path(nome_arquivo), config.pasta_oracao, "application/json")
    print(f"✅ KO classificado: {len(blocos)} bloco(s) (salvo em {nome_arquivo})")

if idiomas_reaproveitados:
    print(f"\nℹ️  {len(idiomas_reaproveitados)} idioma(s) usaram classificação já salva "
          f"(possivelmente corrigida à mão): {', '.join(i.upper() for i in idiomas_reaproveitados)}")

# ── CORREÇÃO AUTOMÁTICA ────────────────────────────────────────────────────
# Roda no que acabou de ser classificado E no que foi reaproveitado do Drive,
# porque as duas fontes têm o mesmo defeito. É idempotente: rodar de novo em
# cima do que já foi corrigido não muda nada.
#
#   corrigir_pontuacao   invariante mecânica (peça só de sinais é pontuação).
#                        No Mateus 2 não corrige nada -- é guarda, não
#                        conserto. Existe porque o erro aqui é INVISÍVEL: a
#                        classe 'outro' do Kiwi sai cinza, igual à pontuação.
#   aplicar_excecoes     o léxico versionado (dados_lexico/classes-excecoes.json).
#                        Começa vazio; enche com o que a célula 6 sugerir.
print("\n" + "─" * 60)
_lexico = carregar_lexico()
print(f"📖 Léxico de exceções: {sum(len(v) for v in _lexico.values())} regra(s)")
for _idioma in list(blocos_por_idioma):
    _blocos, _mud_p = corrigir_pontuacao(blocos_por_idioma[_idioma])
    _blocos, _mud_l = aplicar_excecoes(_blocos, _idioma, _lexico)
    _mudancas = _mud_p + _mud_l
    if _mudancas:
        blocos_por_idioma[_idioma] = _blocos
        print(f"🔧 {_idioma.upper()}: {len(_mudancas)} correção(ões) automática(s)")
        for _m in _mudancas:
            print(f"      {_m}")
        _nome = config.nome_classificacao_multicolor(_idioma)
        salvar_classificacao_multicolor(_blocos, Path(_nome))
        drive_client.upload(Path(_nome), config.pasta_oracao, "application/json")
print("✅ Correção automática aplicada — siga pra célula 5 (revisar)")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  5. REVISAR — a lista de suspeitas, a planilha e a página        ║
# ║  Nada é queimado aqui: esta célula só te mostra o que olhar.     ║
# ╚══════════════════════════════════════════════════════════════════╝
from revisao_classes import suspeitas, exportar_csv, pagina_revisao

# Por que uma lista de suspeitas em vez de "confira tudo": são ~3.300 peças
# nos 5 idiomas. As três regras apontaram 39 delas no Mateus 2 (1,2%), o que
# dá pra ler em cinco minutos:
#
#   sem regra          o analisador devolveu uma etiqueta que o classificador
#                      não trata -- a cor saiu de um chute, não de uma análise
#   classe instável    a mesma palavra recebeu outra classe no resto da legenda
#   classe rara        classe com até 3 ocorrências no idioma inteiro
#
# NÃO é lista de erro: ambiguidade legítima entra junto ("où" é advérbio numa
# frase e pronome na outra). É lista de ONDE olhar.
_suspeitas = suspeitas(blocos_por_idioma)
print(f"🔎 {len(_suspeitas)} peça(s) pra conferir")
_por_idioma = {}
for _s in _suspeitas:
    _por_idioma.setdefault(_s.idioma, []).append(_s)
for _idioma, _lista in sorted(_por_idioma.items()):
    print(f"\n── {_idioma.upper()} ({len(_lista)}) ──")
    for _s in _lista:
        print(f"   bloco {_s.bloco:2d}  «{_s.palavra}» = {_s.classe}")
        print(f"              {_s.motivo}")

# ── Os dois arquivos da revisão ────────────────────────────────────────────
# A página é pra ACHAR o erro: mostra a legenda pintada com as cores de
# verdade, os 5 idiomas empilhados como no vídeo. Nome de classe não é o que
# se enxerga -- o erro aparece quando "Herodes" sai preto no meio de nomes
# próprios amarelos.
# O CSV é pra CORRIGIR: abre no Sheets, você muda a coluna `classe`, e a
# célula 6 lê de volta.
_csv = Path(config.nome_revisao_classes())
_html = Path(config.nome_pagina_revisao_classes())
exportar_csv(blocos_por_idioma, _csv, _suspeitas)
pagina_revisao(blocos_por_idioma, _html, _suspeitas, titulo=NOME_ORACAO)
drive_client.upload(_csv, config.pasta_oracao, "text/csv")
drive_client.upload(_html, config.pasta_oracao, "text/html")
print(f"\n💾 {_csv.name} e {_html.name} — no Drive e aqui embaixo pra baixar")

from google.colab import files
files.download(str(_html))
files.download(str(_csv))
print("\n➡️  Abra o HTML no navegador. Se estiver tudo certo, PULE a célula 6")
print("    e vá direto pra 7 (gerar o .ass).")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  6. APLICAR A CORREÇÃO MANUAL — só se você mexeu no CSV          ║
# ║  Nada mudou na revisão? Pule esta célula.                        ║
# ╚══════════════════════════════════════════════════════════════════╝
from revisao_classes import importar_csv, diferencas, sugerir_excecoes, ErroDeRevisao
from google.colab import files

print("Envie o CSV corrigido (ou cancele pra pular):")
_enviados = files.upload()

if not _enviados:
    print("⏭️  Nada enviado — seguindo com a classificação como está.")
else:
    _nome_csv = list(_enviados.keys())[0]
    # Recusar é melhor que aceitar torto: uma classe com erro de digitação
    # sairia CINZA no vídeo, igualzinho a uma pontuação, e ninguém veria
    # antes de assistir. Por isso o importador levanta em vez de avisar.
    try:
        _corrigido, _resumo = importar_csv(Path(_nome_csv))
    except ErroDeRevisao as _e:
        print(f"❌ {_e}")
        raise SystemExit("Corrija o CSV e rode esta célula de novo.")

    _faltando = [i for i in blocos_por_idioma if i not in _corrigido]
    if _faltando:
        raise SystemExit(f"❌ O CSV não tem estes idiomas: {_faltando}. "
                         f"Envie o arquivo inteiro, não uma aba filtrada.")

    _mudancas = diferencas(blocos_por_idioma, _corrigido)
    print(f"\n✏️  {len(_mudancas)} correção(ões):")
    for _m in _mudancas:
        print(f"   {_m['idioma']} bloco {_m['bloco']:2d}  «{_m['palavra']}»  "
              f"{_m['de']} → {_m['para']}")

    for _idioma in blocos_por_idioma:
        blocos_por_idioma[_idioma] = _corrigido[_idioma]
        _div = classificacao_confere(_corrigido[_idioma], legendas_por_idioma_raw[_idioma])
        if _div:
            print(f"   🚩 {_idioma.upper()}: a correção não reproduz mais o SRT — {_div}")
        _nome = config.nome_classificacao_multicolor(_idioma)
        salvar_classificacao_multicolor(_corrigido[_idioma], Path(_nome))
        drive_client.upload(Path(_nome), config.pasta_oracao, "application/json")
    print("\n💾 Classificação corrigida salva no Drive.")

    # ── Pra correção não morrer com este vídeo ─────────────────────────────
    # O que você corrigiu MAIS DE UMA VEZ provavelmente é defeito do
    # analisador, não do contexto -- e defeito do analisador vai se repetir em
    # Mateus 3, 4, 5. Vira entrada no léxico versionado e nunca mais aparece.
    _sugestao = sugerir_excecoes(_mudancas)
    if _sugestao:
        print("\n" + "═" * 60)
        print("📌 Correções que se REPETIRAM — candidatas a virar automáticas.")
        print("   Preencha o 'porque' e acrescente em")
        print("   pipeline/dados_lexico/classes-excecoes.json (no repositório,")
        print("   não no Drive: o sincronizador sobrescreve o do Drive).")
        print("═" * 60)
        print(_sugestao)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  7. GERAR O .ASS COM CAIXA COLORIDA                              ║
# ╚══════════════════════════════════════════════════════════════════╝
from renderizacao import gerar_ass

config_render = PipelineConfig(NOME_ORACAO=NOME_ORACAO, PASTA_DRIVE_RAIZ=PASTA_DRIVE_RAIZ, IDIOMA_MESTRE=IDIOMA_MESTRE)

caminho_ass = gerar_ass(blocos_por_idioma, config_render, box_border=BOX_BORDER,
                        espaco_colado=ESPACO_ENTRE_PECAS_COLADAS)
print(f"✅ Legenda gerada: {caminho_ass}")
print("\nRode a célula 8 pra baixar e revisar. Só rode a célula 9 (salvar no Drive) depois de confirmar que ficou bom.")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  8. BAIXAR O .ASS (pra revisar / corrigir manualmente)           ║
# ╚══════════════════════════════════════════════════════════════════╝
from google.colab import files
files.download(str(caminho_ass))


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  9. SALVAR NO DRIVE — só rode depois de conferir que ficou bom   ║
# ║  (revise o .ass baixado na célula 8 antes de rodar essa aqui)    ║
# ╚══════════════════════════════════════════════════════════════════╝
caminho_no_drive = drive_client.upload(caminho_ass, config_render.pasta_oracao)
print(f"✅ Salvo no Drive: {caminho_no_drive}")
